# Lab Day 2 — DeepWeeds, notebook điều khiển toàn bộ bài
**Đọc trước:** chạy từng phần theo thứ tự. Chưa chọn Run all lần đầu.
Bài chỉ có kết quả sau khi chạy GPU thật. Có 17 lần train chính, 10 epoch/lần.
Chọn cấu hình hoàn toàn trên val. Test chỉ mở sau khi khóa cấu hình và train đủ 6 lần chung kết.
Mỗi lần train/epoch có checkpoint trên Drive để tiếp tục qua nhiều phiên Colab miễn phí.


## 1. Kết nối Google Drive
Bạn đăng nhập tài khoản chứa thư mục bài lab và cấp quyền khi Colab yêu cầu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Giải nén code và cài môi trường
Các thư viện CUDA của Colab được giữ nguyên. Nếu runtime đã import các gói với phiên bản khác, khởi động lại runtime rồi chạy lại phần 1–2.

In [ ]:
from pathlib import Path
import zipfile, subprocess, sys
DRIVE_ROOT=Path('/content/drive/MyDrive/Lab_Day2_DeepWeeds')
CODE_ROOT=Path('/content/lab_day2_code')
CODE_ROOT.mkdir(exist_ok=True)
bundle=DRIVE_ROOT/'lab_day2_code.zip'
assert bundle.is_file(), f'Thiếu bundle code: {bundle}'
with zipfile.ZipFile(bundle) as z:
    for member in z.infolist():
        target=(CODE_ROOT/member.filename).resolve()
        assert target.is_relative_to(CODE_ROOT.resolve())
    z.extractall(CODE_ROOT)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(CODE_ROOT/'requirements-colab.txt')])
sys.path.insert(0,str(CODE_ROOT))
print('Code root:',CODE_ROOT)


## 3. Kiểm tra môi trường và khởi tạo Study
EDA dùng CPU. Để train, chọn Runtime → Change runtime type → GPU khi tài khoản có GPU khả dụng.

In [ ]:
import torch
from lab_solution.experiments import Study
from lab_solution.common import read_json
STUDY_ROOT=DRIVE_ROOT/'study'
DATA_ROOT=Path('/content/deepweeds')
NOTEBOOK_URL='https://colab.research.google.com/drive/1QrkPHy9DYB2CJkxg9ApFOqUcsGu-PEwj'
# Nếu bạn lưu một bản notebook khác, dán link bản đó vào NOTEBOOK_URL.
study=Study(STUDY_ROOT,DATA_ROOT,notebook_url=NOTEBOOK_URL)
print(study.check_environment())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Chưa có GPU; vẫn chạy được EDA')


## 4. Chuẩn bị ảnh và EDA
Tải từ Zenodo nếu Drive chưa có ZIP. Kiểm tra MD5. Ảnh nằm ở /content để đọc nhanh, bằng chứng lưu trên Drive.

In [ ]:
from lab_solution.preparation import prepare_data, eda
prepare_data(DRIVE_ROOT,DATA_ROOT)
if not (STUDY_ROOT/'eda'/'image_metadata.json').exists():
    eda(DATA_ROOT,STUDY_ROOT/'eda')
else:
    from lab_solution.dataset import load_split, check_split
    check_split(*load_split(DATA_ROOT/'labels'),DATA_ROOT/'images')
    print('EDA đã hoàn tất trước đây, dùng bằng chứng đã lưu.')
from IPython.display import display, Image
for name in ('class_distribution.png','train_samples_3_per_class.png'):
    display(Image(filename=str(STUDY_ROOT/'eda'/name)))


## 5. Kiểm tra tính đúng đắn của code
Test repo gốc kiểm tra starter vẫn là khung. Bộ test mới kiểm tra code hoàn thiện, bao gồm resume/khóa test.
Các số trong unit test là fixture kỹ thuật, không phải kết quả DeepWeeds.


In [ ]:
import os
os.environ['PYTHONIOENCODING']='utf-8'
subprocess.check_call([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=CODE_ROOT)
subprocess.check_call([sys.executable,'-m','unittest','discover','-s','solution_tests','-v'],cwd=CODE_ROOT)


## 6. Kiểm tra pipeline trên ảnh thật
Cần GPU. Overfit 4 ảnh train, CE dưới0.05; xem ảnh augmentation và CutMix. Nếu lỗi, không train thí nghiệm.

In [ ]:
from lab_solution.sanity import pipeline_checks
if not (STUDY_ROOT/'pipeline_checks.json').exists():
    pipeline_checks(DATA_ROOT,STUDY_ROOT,device='cuda')
else:
    from lab_solution.common import read_json
    assert read_json(STUDY_ROOT/'pipeline_checks.json')['passed'], 'Pipeline chưa đạt'
for name in ('overfit_batch.png','augmentation_cutmix.png'):
    display(Image(filename=str(STUDY_ROOT/'pipeline'/name)))


## 7. Preflight cả 5 backbone
Kiểm tra bộ nhớ có tính optimizer Adam. Chốt batch chung16×4 hoặc8×8 trước các thí nghiệm, lưu tag pretrained.

In [ ]:
if not (STUDY_ROOT/'batch_plan.json').exists():
    print(study.preflight())
else:
    print('Đã có batch plan:',read_json(STUDY_ROOT/'batch_plan.json'))


## 8. So sánh backbone — chạy mỗi ô riêng
Mỗi ô có thể chạy lâu. Khi ngắt phiên, chạy lại phần kết nối/chuẩn bị và đúng ô này để resume. Chưa chạy test.

In [ ]:
study.train_backbone('B01')


In [ ]:
study.train_backbone('B02')


In [ ]:
study.train_backbone('B03')


In [ ]:
study.train_backbone('B04')


In [ ]:
study.train_backbone('B05')


In [ ]:
study.select_backbone()


## 9. Ablation — một backbone, một thay đổi mỗi lần
T01 scratch, T02 frozen, T03 CutMix, T04 label smoothing, T05 focal. T06 kết hợp; cần T04/T05 chạy xong trước.

In [ ]:
study.train_ablation('T01')


In [ ]:
study.train_ablation('T02')


In [ ]:
study.train_ablation('T03')


In [ ]:
study.train_ablation('T04')


In [ ]:
study.train_ablation('T05')


In [ ]:
study.train_ablation('T06')


In [ ]:
study.select_recipe()


## 10. So sánh suy luận và độ trễ
I00 mốc; I01 flip; I02 five-crop; I03 AMP; I04 temperature scaling. Đánh giá chỉ trên val, timing trên cùng GPU, batch1/32.

In [ ]:
study.compare_inference()
display(Image(filename=str(STUDY_ROOT/'inference'/'accuracy_latency.png')))


## 11. Khóa cấu hình cuối
Chỉ chạy sau khi đủ backbone/training/inference. Từ đây không quay lại chọn cấu hình bằng test. Chỉ số tốt/xấu đều báo trung thực.

In [ ]:
study.lock_final()


## 12. Chung kết — train đủ 6 lần trước test
Baseline và F01 đều train riêng với seed0/1/2,10epoch. Chạy mỗi ô, resume nếu ngắt. Không dùng kết quả test để chỉnh recipe.

In [ ]:
study.final_train('T00',0)


In [ ]:
study.final_train('T00',1)


In [ ]:
study.final_train('T00',2)


In [ ]:
study.final_train('F01',0)


In [ ]:
study.final_train('F01',1)


In [ ]:
study.final_train('F01',2)


## 13. Test — một lượt hoàn tất mỗi cấu hình/seed
Phần đã lưu sẽ không suy luận lại. Bản uncal dùng cùng logit test, không chạy test thêm. Không thay đổi cấu hình sau khi mở kết quả.

In [ ]:
study.final_test('T00',0)


In [ ]:
study.final_test('T00',1)


In [ ]:
study.final_test('T00',2)


In [ ]:
study.final_test('F01',0)


In [ ]:
study.final_test('F01',1)


In [ ]:
study.final_test('F01',2)


## 14. Xuất Excel, biểu đồ lỗi và báo cáo bản đầu
Chỉ hoạt động khi đủ dự đoán thật. eval.py gốc tính lại score/grade; Excel7sheet. Báo cáo cần nhận xét quan sát ảnh của bạn.

In [ ]:
study.export()
display(Image(filename=str(STUDY_ROOT/'analysis'/'F01_confusion.png')))
if (STUDY_ROOT/'analysis'/'error_images_seed0.png').exists():
    display(Image(filename=str(STUDY_ROOT/'analysis'/'error_images_seed0.png')))


## 15. Viết nhận xét bằng chứng (không bịa)
Điền vào ô bên dưới sau khi xem các bảng/ảnh. Đề cập: EDA, hội tụ/overfit, Δ ablation, tác dụng kết hợp, yếu tố đóng góp nhiều nhất, lỗi Chinee apple/Snake weed, triển khai30–100ms, hạn chế. Ít nhất200ký tự để tránh đóng gói bản chưa phân tích.


In [ ]:
ANALYSIS_NOTES = """"""
# Điền nhận xét thật giữa dấu nháy ba phía trên.
if len(ANALYSIS_NOTES.strip())>=200:
    (STUDY_ROOT/'analysis_notes.md').write_text(ANALYSIS_NOTES,encoding='utf-8')
    study.export()
else:
    print('Chưa ghi đè nhận xét. Cần viết nhận xét thật trước khi đóng gói.')


## 16. Kiểm tra trên phiên Colab mới
Sau khi đủ kết quả và nhận xét, khởi động lại runtime. Chạy lại setup, cài môi trường, tạo Study và chuẩn bị ảnh; không chạy lại thí nghiệm/chọn cấu hình/test. Sau đó chạy ô dưới. Nó nạp checkpoint và kiểm tra compatibility, không suy luận lại test.

In [ ]:
study.verify_fresh_session()
study.export()
status=study.validate()
print(status)
if not status["ready_to_submit"]:
    print("Cần bổ sung các mục errors trước khi đóng gói.")


## 17. Đóng gói bài nộp
Điền MSSV và họ tên không dấu, dùng dấu gạch dưới thay khoảng trắng. Chỉ đóng gói khi study.validate() đạt; ZIP không chứa dataset/checkpoint lớn. Không tự gửi bài hoặc tạo PR.

In [ ]:
STUDENT_ID = ''
STUDENT_NAME = ''
if STUDENT_ID and STUDENT_NAME:
    result=study.export(student_id=STUDENT_ID,student_name=STUDENT_NAME)
    print(result)
else:
    print('Điền MSSV và họ tên không dấu sau khi hoàn thành nhận xét.')


## Sau ngắt phiên
Chạy lại ô1–4 để kết nối Drive, cài đúng môi trường và chuẩn bị dữ liệu, rồi tạo lại Study. Không chạy lại các ô chọn cấu hình sau khi đã có final_lock. Đi tới đúng ô train/test bị ngắt; checkpoint và file dự đoán đã hoàn tất được tái sử dụng.
Mọi số liệu phải có nguồn log/exp_id/seed. Hiệu chuẩn luôn dùng val. Có kết quả thấp vẫn phân tích trung thực, không sửa eval.py hoặc CSV.
